# Phase 2 — Smoke Test v1 (QLoRA Qwen3.5-4B-Base)

**Mục đích:** Fine-tune 20K samples, validate pipeline end-to-end trước khi chạy full data (Phase 3).  
**Design:** Section 4.5 `plan_day5.md` — 8 decisions (Q1-Q8) + 8 refinements (R1-R8). File này READ-ONLY với Sonnet.  
**Framework:** PEFT + bitsandbytes 4-bit NF4 (không Unsloth — chốt 2026-04-26).  

| Param | Value |
|-------|-------|
| Model | `Qwen/Qwen3.5-4B-Base` |
| Dataset | `SeanSunny/items_prompts_tv_3` |
| Train size | 20,000 samples |
| LoRA | r=32, alpha=64, attention-only |
| Epochs | 2 |
| Max seq length | 192 |

**Kỳ vọng RMSLE:** < 0.6 (kỳ vọng), < 4.44 (minimum — phải beat v0 zero-shot).

## 0. Cài đặt (chạy một lần sau khi git clone + uv sync)

```bash
uv sync
uv add "transformers>=5.2.0"  # bat buoc cho Qwen3.5
```

Cần: `transformers>=5.2.0`, `peft`, `trl`, `bitsandbytes`, `accelerate`, `datasets`, `python-dotenv` — đã có trong `pyproject.toml`.

In [ ]:
# Chay neu chua co trong env:
#!uv add "transformers>=5.2.0" peft trl bitsandbytes accelerate datasets python-dotenv

Resolved 284 packages in 1ms
Checked 278 packages in 31ms


In [2]:
import os                                                                                                                                                 
import re                                                 
import sys
import json
import time
import glob                                                                                                                                               
import math
import numpy as np                                                                                                                                        
from tqdm import tqdm                                     
from pathlib import Path

import torch
import bitsandbytes as bnb
import torch.nn as nn                                                                                                                                     
from datasets import load_dataset
from dotenv import load_dotenv                                                                                                                            
from huggingface_hub import login                                                                                                                         
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig                                                                          
from trl import SFTTrainer, SFTConfig  # DataCollatorForCompletionOnlyLM removed in 0.24.0
                                                                                                                                                        
from dataclasses import dataclass                         
from typing import Any, Dict, List                                                                                                                        
from transformers import PreTrainedTokenizerBase          
                                                                                                                                                        
@dataclass
class DataCollatorForCompletionOnlyLM:                                                                                                                    
    """Manual impl: trl.DataCollatorForCompletionOnlyLM removed in TRL 0.24.0."""
    response_template: List[int]                                                                                                                          
    tokenizer: PreTrainedTokenizerBase                                                                                                                    
    ignore_index: int = -100                                                                                                                              
                                                                                                                                                        
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids_list = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        max_len = max(len(x) for x in input_ids_list)                                                                                                     
        bs = len(input_ids_list)
                                                                                                                                                        
        padded = torch.full((bs, max_len), self.tokenizer.pad_token_id, dtype=torch.long)
        attn   = torch.zeros((bs, max_len), dtype=torch.long)                                                                                             
        labels = torch.full((bs, max_len), self.ignore_index, dtype=torch.long)                                                                           

        tpl, tpl_len = self.response_template, len(self.response_template)                                                                                
                                                        
        for i, ids in enumerate(input_ids_list):                                                                                                          
            n = len(ids)
            padded[i, :n] = ids                                                                                                                           
            attn[i, :n]   = 1                             
            for j in range(n - tpl_len, -1, -1):
                if ids[j : j + tpl_len].tolist() == tpl:                                                                                                  
                    labels[i, j + tpl_len : n] = ids[j + tpl_len : n]
                    break                                                                                                                                 
                                                        
        return {"input_ids": padded, "attention_mask": attn, "labels": labels}                                                                            
                                                        
print("DataCollatorForCompletionOnlyLM: manual impl OK")                                                                                                  

NOTEBOOK_DIR = Path("__file__").parent if "__file__" in dir() else Path(".")                                                                              
sys.path.insert(0, str(NOTEBOOK_DIR))                     
from utils.evaluator import compute_metrics, plot_predictions
                                                                                                                                                        
print("Imports OK")
import transformers                                                                                                                                       
import peft                                               
import trl
for pkg, mod in [("torch", torch), ("transformers", transformers), ("peft", peft), ("trl", trl)]:
    print(f"  {pkg:<14}: {mod.__version__}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+cu128).


DataCollatorForCompletionOnlyLM: manual impl OK
Imports OK
  torch         : 2.9.0+cu128
  transformers  : 5.5.0
  peft          : 0.19.1
  trl           : 0.24.0


In [3]:
# --- Section 4.5.3 constants (chot, khong sua) ---

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME = "SeanSunny/items_prompts_tv_3"

# Sequence
MAX_SEQ_LENGTH  = 192
MAX_NEW_TOKENS  = 4
QUESTION_PREFIX = "S\u1ea3n ph\u1ea9m n\u00e0y c\u00f3 gi\u00e1 bao nhi\u00eau ?\n"
PRICE_PREFIX    = "\n\nGi\u00e1 l\u00e0: "

# LoRA (smoke v1)
LORA_R              = 32
LORA_ALPHA          = 64
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training (smoke)
TRAIN_SIZE        = 20000
VAL_EVAL_SIZE     = 500
NUM_EPOCHS        = 2
PER_DEVICE_BATCH  = 8
GRAD_ACCUM        = 8       # effective batch = 64
LEARNING_RATE     = 2e-4
LR_SCHEDULER      = "cosine"
WARMUP_RATIO      = 0.03
WEIGHT_DECAY      = 0.001
OPTIM             = "paged_adamw_32bit"
PACKING           = False
GRADIENT_CHECKPOINTING = True
EVAL_STEPS        = 100
SAVE_STRATEGY     = "epoch"
LOGGING_STEPS     = 20
SEED              = 42

# Inference safety (Q5 lai)
PRED_CLAMP_MIN = 5
PRED_CLAMP_MAX = 1000
PARSE_REGEX    = r"[-+]?\d*\.\d+|\d+"

# Paths
ADAPTER_DIR  = NOTEBOOK_DIR / "weights" / "v1_adapter"
RESULTS_FILE = NOTEBOOK_DIR / "results" / "v1_results.json"
HF_REPO_ADAPTER = "SeanSunny/qwen3.5-4b-vn-pricer-v1"

print(f"BASE_MODEL    : {BASE_MODEL}")
print(f"DATASET_NAME  : {DATASET_NAME}")
print(f"ADAPTER_DIR   : {ADAPTER_DIR}")
print(f"RESULTS_FILE  : {RESULTS_FILE}")

BASE_MODEL    : Qwen/Qwen3.5-4B-Base
DATASET_NAME  : SeanSunny/items_prompts_tv_3
ADAPTER_DIR   : weights/v1_adapter
RESULTS_FILE  : results/v1_results.json


In [4]:
# GPU check + HF login
assert torch.cuda.is_available(), "GPU khong kha dung."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
cap = torch.cuda.get_device_capability()
print(f"bf16 : {'yes' if cap[0] >= 8 else 'no'} (compute {cap})")

env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print(f"HF login OK (from {env_path})")
else:
    print(f"HF_TOKEN not set in {env_path}")
    login()

# Create output dirs
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
(NOTEBOOK_DIR / "results").mkdir(exist_ok=True)
print("Dirs ready: weights/v1_adapter/, results/")

GPU  : NVIDIA GeForce RTX 3090 Ti
VRAM : 25.3 GB
bf16 : yes (compute (8, 6))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK (from .env)
Dirs ready: weights/v1_adapter/, results/


## 1. Load model + tokenizer

BitsAndBytesConfig 4-bit NF4 + double quant.  
**R4:** `prepare_model_for_kbit_training` TRƯỚC `get_peft_model` — đúng thứ tự.  
**R5:** Verify EOS/PAD tokens Qwen3.5-Base.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# R5: verify EOS/PAD
if tokenizer.eos_token_id is None:
    tokenizer.eos_token = "<|endoftext|>"
    print("WARNING: set eos_token manually to <|endoftext|>")
print(f"EOS token : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"PAD token : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# R4: prepare TRUOC khi apply LoRA
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING)
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

EOS token : '<|endoftext|>' (id=248044)
PAD token : '<|endoftext|>' (id=248044)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Memory footprint: 4.33 GB


## 2. Verify Qwen3.5 modules + apply LoRA

**R2:** Qwen3.5 co architecture Hybrid (Gated DeltaNet + sparse MoE) — module names can khac.  
In tat ca Linear suffixes truoc khi apply LoRA. Fallback `all-linear` neu thieu attention modules.

In [6]:
linear_suffixes = set()
for name, module in model.named_modules():
    if isinstance(module, (nn.Linear, bnb.nn.Linear4bit)):
        linear_suffixes.add(name.split(".")[-1])

print("Linear module suffixes found:", sorted(linear_suffixes))

EXPECTED_ATTN = {"q_proj", "k_proj", "v_proj", "o_proj"}
if EXPECTED_ATTN.issubset(linear_suffixes):
    target_modules = LORA_TARGET_MODULES
    print(f"PASS: attention modules found. target_modules = {target_modules}")
else:
    target_modules = "all-linear"
    missing = EXPECTED_ATTN - linear_suffixes
    print(f"WARNING: missing {missing}. Fallback target_modules = 'all-linear'")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# enable_input_require_grads — bao ve truong hop loss = 0 tu step 1
model.enable_input_require_grads()

Linear module suffixes found: ['down_proj', 'gate_proj', 'in_proj_a', 'in_proj_b', 'in_proj_qkv', 'in_proj_z', 'k_proj', 'lm_head', 'o_proj', 'out_proj', 'q_proj', 'up_proj', 'v_proj']
PASS: attention modules found. target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
trainable params: 6,291,456 || all params: 4,212,042,752 || trainable%: 0.1494


## 3. Dataset + truncation analysis

**Q1=A:** Tinh TOKENS_FIXED runtime (QUESTION_PREFIX + PRICE_PREFIX tokens).  
**R7:** Log p50/p95/p99 summary token len + truncation rate (ky vong < 30%).

In [7]:
ds = load_dataset(DATASET_NAME)
print(f"Train: {len(ds['train']):,} | Val: {len(ds['val']):,} | Test: {len(ds['test']):,}")

# Q1=A: compute TOKENS_FIXED runtime — khong hardcode
q_ids = tokenizer.encode(QUESTION_PREFIX, add_special_tokens=False)
p_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
TOKENS_FIXED = len(q_ids) + len(p_ids)

# completion max = 4 tokens (profile_results_v3.json), EOS = 1, safety_buffer = 2
MAX_SUMMARY_TOKENS = MAX_SEQ_LENGTH - TOKENS_FIXED - MAX_NEW_TOKENS - 1 - 2

print(f"QUESTION_PREFIX tokens : {len(q_ids)}")
print(f"PRICE_PREFIX tokens    : {len(p_ids)}")
print(f"TOKENS_FIXED           : {TOKENS_FIXED}")
print(f"MAX_SUMMARY_TOKENS     : {MAX_SUMMARY_TOKENS}")
print(f"  (= {MAX_SEQ_LENGTH} - {TOKENS_FIXED} - {MAX_NEW_TOKENS} - 1 - 2)")

# Lay 20K train sample + 500 val raw (truoc preprocess)
train_sample_raw = ds["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_raw = ds["val"].shuffle(seed=SEED).select(range(VAL_EVAL_SIZE))

# R7: do distribution summary token len tren tap train
summaries = []
for item in train_sample_raw:
    p = item["prompt"]
    summaries.append(p[len(QUESTION_PREFIX):-len(PRICE_PREFIX)])

summary_lens = np.array(
    [len(tokenizer.encode(s, add_special_tokens=False)) for s in tqdm(summaries, desc="Tokenizing summaries")]
)
n_truncated = int((summary_lens > MAX_SUMMARY_TOKENS).sum())

print(f"\nSummary token len (train {TRAIN_SIZE:,}):")
print(f"  p50={np.percentile(summary_lens, 50):.0f}, "
      f"p95={np.percentile(summary_lens, 95):.0f}, "
      f"p99={np.percentile(summary_lens, 99):.0f}, "
      f"max={summary_lens.max()}")
print(f"Truncated: {n_truncated}/{len(summaries)} ({n_truncated/len(summaries)*100:.1f}%)")
if n_truncated / len(summaries) > 0.5:
    print("WARNING: truncation rate > 50%. MAX_SUMMARY_TOKENS co the qua nho — xem lai profile.")

README.md:   0%|          | 0.00/556 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/791k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/780k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/85727 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/3926 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3872 [00:00<?, ? examples/s]

Train: 85,727 | Val: 3,926 | Test: 3,872
QUESTION_PREFIX tokens : 9
PRICE_PREFIX tokens    : 5
TOKENS_FIXED           : 14
MAX_SUMMARY_TOKENS     : 171
  (= 192 - 14 - 4 - 1 - 2)


Tokenizing summaries: 100%|██████████| 20000/20000 [00:06<00:00, 3021.13it/s]


Summary token len (train 20,000):
  p50=98, p95=128, p99=148, max=228
Truncated: 21/20000 (0.1%)


In [8]:
# Preprocess: cat summary token-level tu duoi (Q2/R7), build full_text voi EOS
def preprocess(example):
    p = example["prompt"]
    summary = p[len(QUESTION_PREFIX):-len(PRICE_PREFIX)]
    summary_ids = tokenizer.encode(summary, add_special_tokens=False)
    if len(summary_ids) > MAX_SUMMARY_TOKENS:
        summary_ids = summary_ids[:MAX_SUMMARY_TOKENS]
        summary = tokenizer.decode(summary_ids, skip_special_tokens=True).rstrip()
    # Q5: them "\n" + eos_token sau completion (model hoc stop ro rang hon)
    full_text = QUESTION_PREFIX + summary + PRICE_PREFIX + example["completion"] + "\n" + tokenizer.eos_token
    return {"text": full_text}

train_ds = train_sample_raw.map(preprocess, desc="Preprocess train")
val_ds   = val_raw.map(preprocess, desc="Preprocess val")

print(f"train_ds: {len(train_ds):,} | val_ds: {len(val_ds):,}")
print()

# Eyeball 2 samples: kiem tra PRICE_PREFIX con nguyen + completion dung
for i in range(2):
    t = train_ds[i]["text"]
    print(f"--- Sample {i} (len={len(t)}) ---")
    print(repr(t[:250]) + ("..." if len(t) > 250 else ""))
    assert PRICE_PREFIX in t, f"ERROR: PRICE_PREFIX missing in sample {i}"
    print()

print("PRICE_PREFIX present in all shown samples: OK")

Preprocess train:   0%|          | 0/20000 [00:00<?, ? examples/s]

Preprocess val:   0%|          | 0/500 [00:00<?, ? examples/s]

train_ds: 20,000 | val_ds: 500

--- Sample 0 (len=317) ---
'Sản phẩm này có giá bao nhiêu ?\nTiêu đề: Giày tây nam lịch lãm NA18  \nDanh mục: Giày nam  \nThương hiệu: Giày nam  \nMô tả: Giày da thật màu đen, thiết kế buộc dây đơn giản, phù hợp cho phong cách lịch lãm.  \nThông số: Đế cao su 3cm, da thuộc mềm chịu '...

--- Sample 1 (len=342) ---
'Sản phẩm này có giá bao nhiêu ?\nTiêu đề: Combo Dầu Gội Khô Colab Dark & Tropical 200ml  \nDanh mục: Dầu gội khô & chăm sóc tóc  \nThương hiệu: Colab  \nMô tả: Dưỡng tóc nhẹ nhàng, thấm hút dầu hiệu quả, không để lại vệt trắng.  \nThông số: Công nghệ tia '...

PRICE_PREFIX present in all shown samples: OK


## 4. DataCollator + verify mask

**R1:** `DataCollatorForCompletionOnlyLM` phai dung TOKEN IDS, khong phai string (BPE merge gotcha).  
**R6:** Verify labels fail-loud — neu mask sai thi dung notebook, khong train.

In [12]:
# TRL 0.24.0: khong con dataset_text_field — pre-tokenize truoc khi truyen vao trainer
def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

train_ds_tok = train_ds.map(tokenize_fn, batched=False, remove_columns=["text"])
val_ds_tok   = val_ds.map(tokenize_fn, batched=False, remove_columns=["text"])

print(f"Columns: {train_ds_tok.column_names}")
print(f"Sample input_ids len: {len(train_ds_tok[0]['input_ids'])}")

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Columns: ['prompt', 'completion', 'price_vnd_true', 'input_ids', 'attention_mask']
Sample input_ids len: 103


In [13]:
# R1: encode PRICE_PREFIX thanh token IDs
response_template_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
decoded_back = tokenizer.decode(response_template_ids)
print(f"response_template_ids : {response_template_ids}")
print(f"decoded back          : {decoded_back!r}")
print(f"matches PRICE_PREFIX  : {decoded_back == PRICE_PREFIX}")

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,  # LIST INT, khong phai str
    tokenizer=tokenizer,
)

# R6: verify mask fail-loud
sample_text = train_ds[0]["text"]
tokenized = tokenizer(
    sample_text,
    return_tensors="pt",
    max_length=MAX_SEQ_LENGTH,
    truncation=True,
)
batch_in = [{
    "input_ids": tokenized["input_ids"][0].tolist(),
    "attention_mask": tokenized["attention_mask"][0].tolist(),
}]
batch_out = collator(batch_in)
labels = batch_out["labels"][0]
non_masked = labels[labels != -100]
decoded_labels = tokenizer.decode(non_masked.tolist(), skip_special_tokens=False)

print(f"\nNon-masked token count : {len(non_masked)}")
print(f"Decoded non-masked     : {decoded_labels!r}")
print(f"Expected               : completion + '\\n' + eos_token")
print(f"Sample completion      : {train_sample_raw[0]['completion']!r}")

# Fail-loud: neu non_masked rong -> response_template_ids khong match -> dung
if len(non_masked) == 0:
    raise RuntimeError(
        "Mask verify FAILED: response_template_ids not found in tokenized sequence. "
        "PRICE_PREFIX encoding co the bi BPE merge khac khi dung trong context. "
        "Abort training."
    )

# Kiem tra khong co prompt tokens bi leak vao labels
expected_max_tokens = MAX_NEW_TOKENS + 2  # completion + \n + eos
if len(non_masked) > expected_max_tokens + 2:  # +2 buffer
    print(f"WARNING: non-masked count {len(non_masked)} > expected {expected_max_tokens}. "
          f"Co the co prompt tokens bi leak. Kiem tra lai response_template_ids.")
else:
    print("\nMask verify PASS: chi thay completion + EOS trong labels.")

response_template_ids : [271, 185394, 36663, 25, 220]
decoded back          : '\n\nGiá là: '
matches PRICE_PREFIX  : True

Non-masked token count : 5
Decoded non-masked     : '420\n<|endoftext|>'
Expected               : completion + '\n' + eos_token
Sample completion      : '420'

Mask verify PASS: chi thay completion + EOS trong labels.


## 5. VRAM smoke (100 samples)

**R8:** Train 100 samples, max_steps=5 — do VRAM peak + sec/step, uoc tinh total time.  
OOM fallback: giam `PER_DEVICE_BATCH` xuong 4 + tang `GRAD_ACCUM` len 16.

In [16]:
torch.cuda.reset_peak_memory_stats()
t_smoke_start = time.time()                                                                                                                               
                                                        
trainer_smoke = SFTTrainer(
    model=model,
    processing_class=tokenizer,                                                                                                                                  
    train_dataset=train_ds_tok.select(range(100)),  # dung _tok
    data_collator=collator,                                                                                                                               
    args=SFTConfig(                                                                                                                                       
        output_dir=str(NOTEBOOK_DIR / "weights" / "v1_smoke_run"),
        max_steps=5,                                                                                                                                      
        per_device_train_batch_size=PER_DEVICE_BATCH,     
        gradient_accumulation_steps=GRAD_ACCUM,                                                                                                           
        learning_rate=LEARNING_RATE,
        bf16=True,                                                                                                                                        
        logging_steps=1,                                  
        save_strategy="no",                                                                                                                               
        report_to="none",
        seed=SEED,                                                                                                                                        
        gradient_checkpointing=GRADIENT_CHECKPOINTING,    
    ),
)
trainer_smoke.train()
                                                                                                                                                        
vram_smoke_gb = torch.cuda.max_memory_allocated() / 1e9
t_smoke = time.time() - t_smoke_start                                                                                                                     
sec_per_step = t_smoke / 5                                

steps_per_epoch = math.ceil(TRAIN_SIZE / (PER_DEVICE_BATCH * GRAD_ACCUM))                                                                                 
total_steps = steps_per_epoch * NUM_EPOCHS
est_total_sec = sec_per_step * total_steps                                                                                                                
                                                                                                                                                        
print(f"\nVRAM peak   : {vram_smoke_gb:.2f} GB")
print(f"Sec/step    : {sec_per_step:.2f}s")                                                                                                               
print(f"Steps/epoch : {steps_per_epoch}")                                                                                                                 
print(f"Total steps : {total_steps}")
print(f"Est. total  : {est_total_sec/60:.1f} min")                                                                                                        
                                                                                                                                                        
if vram_smoke_gb > 22.0:
    print(f"\nWARNING: VRAM peak {vram_smoke_gb:.1f} GB > 22 GB.")                                                                                        
    print("  Fallback: PER_DEVICE_BATCH=4, GRAD_ACCUM=16.")
else:                                                                                                                                                     
    print(f"\nVRAM OK ({vram_smoke_gb:.1f} GB < 22 GB).")
                                                                                                                                                        
print("\n[CHECKPOINT] Smoke OK. Proceed to full train cell.")

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 248044}.


Step,Training Loss
1,4.104156
2,3.260548
3,2.593559
4,2.096036
5,1.834538



VRAM peak   : 7.46 GB
Sec/step    : 18.68s
Steps/epoch : 313
Total steps : 626
Est. total  : 194.8 min

VRAM OK (7.5 GB < 22 GB).

[CHECKPOINT] Smoke OK. Proceed to full train cell.


In [17]:
del trainer_smoke                                                                                                                                         
del smoke_ds                                                                                                                                              
import gc                                                                                                                                                 
gc.collect()                                              
torch.cuda.empty_cache()
                                                                                                                                                        
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")                                                                      
print(f"VRAM reserved     : {torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")

VRAM after cleanup: 3.16 GB allocated
VRAM reserved     : 3.27 GB reserved


## 6. Full train 20K

**Q4 B+:** `eval_strategy="steps"` CE loss in-train.  
**Q8:** `report_to="none"` — console only, khong wandb cho smoke.  
`save_strategy="epoch"` — luu checkpoint cuoi moi epoch de eval per-epoch sau.

In [18]:
torch.cuda.reset_peak_memory_stats()
t_train_start = time.time()                                                                                                                               

trainer = SFTTrainer(                                                                                                                                     
    model=model,                                          
    processing_class=tokenizer,
    train_dataset=train_ds_tok,   # dung _tok                                                                                                             
    eval_dataset=val_ds_tok,      # dung _tok
    data_collator=collator,                                                                                                                               
    args=SFTConfig(                                                                                                                                       
        output_dir=str(ADAPTER_DIR),
        #per_device_train_batch_size=PER_DEVICE_BATCH,                                                                                                     
        per_device_train_batch_size=16,                                                                                                     
        per_device_eval_batch_size=1,                     
        #gradient_accumulation_steps=GRAD_ACCUM,                                                                                                           
        gradient_accumulation_steps=4,                                                                                                           
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,                                                                                                                      
        lr_scheduler_type=LR_SCHEDULER,                   
        warmup_ratio=WARMUP_RATIO,                                                                                                                        
        weight_decay=WEIGHT_DECAY,
        optim=OPTIM,                                                                                                                                      
        bf16=True,                                        
        max_grad_norm=0.3,                                                                                                                                
        #gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing=False,
        eval_strategy="steps",                                                                                                                            
        eval_steps=EVAL_STEPS,                            
        save_strategy=SAVE_STRATEGY,
        save_total_limit=3,                                                                                                                               
        logging_steps=LOGGING_STEPS,
        report_to="none",                                                                                                                                 
        seed=SEED,                                        
    ),
)
trainer.train()
trainer.save_model(str(ADAPTER_DIR))                                                                                                                      
tokenizer.save_pretrained(str(ADAPTER_DIR))
                                                                                                                                                        
total_train_sec = time.time() - t_train_start                                                                                                             
final_vram_peak = torch.cuda.max_memory_allocated() / 1e9
print(f"Training complete: {total_train_sec/60:.1f} min | VRAM peak: {final_vram_peak:.2f} GB")                                                           
                                                                                                                                                        
log_history = trainer.state.log_history                                                                                                                   
train_losses = [(int(e["step"]), float(e["loss"])) for e in log_history if "loss" in e and "eval_loss" not in e]                                          
eval_losses  = [(int(e["step"]), float(e["eval_loss"])) for e in log_history if "eval_loss" in e]                                                         
                                                                                                                                                        
print(f"Train loss entries: {len(train_losses)} | Eval loss entries: {len(eval_losses)}")                                                                 
if train_losses:                                                                                                                                          
    print(f"Train loss: first={train_losses[0][1]:.4f} -> last={train_losses[-1][1]:.4f}")                                                                
if eval_losses:                                                                                                                                           
    print(f"Eval CE loss: first={eval_losses[0][1]:.4f} -> last={eval_losses[-1][1]:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Truncating train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.207301,1.203564,1.919758,761619.000000,0.559633
200,1.171503,1.186069,1.927375,1523425.000000,0.570700
300,1.165524,1.167851,1.923389,2285784.000000,0.578100
400,1.125267,1.163224,2.024965,3045334.000000,0.580833
500,1.113375,1.157596,2.018799,3807090.000000,0.584400
600,1.120750,1.153790,2.020059,4569858.000000,0.585533
626,1.106752,1.154789,2.020505,4763736.000000,0.584033


Training complete: 162.4 min | VRAM peak: 12.81 GB
Train loss entries: 31 | Eval loss entries: 7
Train loss: first=1.4153 -> last=1.1068
Eval CE loss: first=1.2036 -> last=1.1548


## 7. Generative eval — 500 val (final)

**Q5 lai:** Regex float-first `r"[-+]?\d*\.\d+|\d+"` + clamp `[5, 1000]`.  
**Q6:** Chỉ 500 val — full 3872 test để dành Phase 3.

In [21]:
model.eval()

def predict_one(prompt: str) -> tuple:
    """Returns (pred_thousands_vnd, raw_generated_text)."""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(PARSE_REGEX, gen)
    if m:
        pred_k = int(float(m.group()))
        pred_k = max(PRED_CLAMP_MIN, min(pred_k, PRED_CLAMP_MAX))
    else:
        pred_k = 0
    return pred_k, gen

preds_vnd, trues_vnd, raw_outs = [], [], []
clamp_count = 0
t_eval_start = time.time()

for item in tqdm(val_raw, desc="Generative eval val"):
    pred_k, raw = predict_one(item["prompt"])
    preds_vnd.append(pred_k * 1000)
    trues_vnd.append(item["price_vnd_true"])
    raw_outs.append(raw)
    if pred_k in (PRED_CLAMP_MIN, PRED_CLAMP_MAX):
        clamp_count += 1

t_eval = time.time() - t_eval_start
metrics_final = compute_metrics(np.array(trues_vnd, dtype=float), np.array(preds_vnd, dtype=float))

print("=" * 50)
print(f"v1 Smoke — {VAL_EVAL_SIZE} val (epoch {NUM_EPOCHS} final)")
print("=" * 50)
print(f"RMSLE : {metrics_final['rmsle']:.4f}  (primary)")
print(f"MAE   : {metrics_final['mae']:,.0f} VND")
print(f"MAPE  : {metrics_final['mape']:.1f}%")
print(f"R2    : {metrics_final['r2']:.4f}")
print(f"Zero preds   : {preds_vnd.count(0)}")
print(f"Clamp trigger: {clamp_count}")
print(f"Sec/item     : {t_eval/VAL_EVAL_SIZE:.2f}s")
print("=" * 50)
print(f"v0 reference : RMSLE=4.4428")
print(f"Day4 v8 ref  : RMSLE=0.4004")

# Plot (day4 style)
names = [item["prompt"][:50] for item in val_raw]
plot_predictions(
    np.array(trues_vnd, dtype=float),
    np.array(preds_vnd, dtype=float),
    title=f"v1 Smoke ({VAL_EVAL_SIZE} val)",
    names=names,
)

Generative eval val: 100%|██████████| 500/500 [07:04<00:00,  1.18it/s]


v1 Smoke — 500 val (epoch 2 final)
RMSLE : 0.6084  (primary)
MAE   : 116,769 VND
MAPE  : 50.4%
R2    : 0.3980
Zero preds   : 0
Clamp trigger: 0
Sec/item     : 0.85s
v0 reference : RMSLE=4.4428
Day4 v8 ref  : RMSLE=0.4004


## 8. Manual checkpoint eval per-epoch

**Q4 bonus / Q3=A:** Glob auto-detect checkpoint dirs, khong hardcode step count.  
Load adapter tung epoch tren cung quantized base model → so sanh RMSLE epoch 1 vs epoch 2.

In [22]:
# Q3=A: glob auto-detect, sort by step number
checkpoint_dirs = sorted(
    glob.glob(str(ADAPTER_DIR / "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1]),
)
print(f"Checkpoints found: {len(checkpoint_dirs)}")
for d in checkpoint_dirs:
    print(f"  {d}")

metrics_per_epoch = {}

for ep_idx, ckpt_dir in enumerate(checkpoint_dirs):
    ep_num = ep_idx + 1
    print(f"\nEval checkpoint {Path(ckpt_dir).name} (epoch {ep_num})...")
    # Load adapter tren cung quantized base (shared memory, chi swap LoRA weights)
    ckpt_model = PeftModel.from_pretrained(model.base_model.model, ckpt_dir)
    ckpt_model.eval()

    preds_ckpt, trues_ckpt = [], []
    for item in tqdm(val_raw, desc=f"Epoch {ep_num}", leave=False):
        inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
        with torch.inference_mode():
            out = ckpt_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        match = re.search(PARSE_REGEX, gen)
        if match:
            pk = max(PRED_CLAMP_MIN, min(int(float(match.group())), PRED_CLAMP_MAX))
        else:
            pk = 0
        preds_ckpt.append(pk * 1000)
        trues_ckpt.append(item["price_vnd_true"])

    m_ep = compute_metrics(np.array(trues_ckpt, dtype=float), np.array(preds_ckpt, dtype=float))
    metrics_per_epoch[f"epoch_{ep_num}"] = m_ep
    print(f"  Epoch {ep_num}: RMSLE={m_ep['rmsle']:.4f}, MAE={m_ep['mae']:,.0f}")
    del ckpt_model

print("\nEpoch comparison:")
for ep_key, m in metrics_per_epoch.items():
    print(f"  {ep_key}: RMSLE={m['rmsle']:.4f}, MAE={m['mae']:,.0f}, MAPE={m['mape']:.1f}%")

Checkpoints found: 2
  weights/v1_adapter/checkpoint-313
  weights/v1_adapter/checkpoint-626

Eval checkpoint checkpoint-313 (epoch 1)...


/workspace/hieu/DACN3/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning:

Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!

/workspace/hieu/DACN3/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning:

Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!



  Epoch 1: RMSLE=0.6295, MAE=122,374

Eval checkpoint checkpoint-626 (epoch 2)...


  Epoch 2: RMSLE=0.6084, MAE=116,769

Epoch comparison:
  epoch_1: RMSLE=0.6295, MAE=122,374, MAPE=50.5%
  epoch_2: RMSLE=0.6084, MAE=116,769, MAPE=50.4%


## 9. Save + push

**Q7 / Q4=A:** `push_to_hub` truc tiep. Ghi day du v1_results.json (schema plan 4.5.7 cell 12).

In [23]:
# Save v1_results.json
samples_out = []
for i in range(min(20, len(val_raw))):
    tv = trues_vnd[i]
    pv = preds_vnd[i]
    err_pct = abs(pv - tv) / tv * 100 if tv > 0 else None
    samples_out.append({
        "idx": i,
        "prompt_excerpt": val_raw[i]["prompt"][:120],
        "generated_raw": raw_outs[i],
        "pred_vnd": pv,
        "true_vnd": tv,
        "error_pct": round(err_pct, 1) if err_pct is not None else None,
    })

results = {
    "version": "v1_smoke",
    "model": BASE_MODEL,
    "dataset": DATASET_NAME,
    "config": {
        "train_size": TRAIN_SIZE,
        "val_eval_size": VAL_EVAL_SIZE,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "num_epochs": NUM_EPOCHS,
        "per_device_batch": PER_DEVICE_BATCH,
        "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "max_seq_length": MAX_SEQ_LENGTH,
        "tokens_fixed": TOKENS_FIXED,
        "max_summary_tokens": MAX_SUMMARY_TOKENS,
    },
    "truncation_rate_pct": round(n_truncated / len(summaries) * 100, 1),
    "vram_smoke_gb": round(vram_smoke_gb, 2),
    "vram_train_peak_gb": round(final_vram_peak, 2),
    "total_train_sec": round(total_train_sec, 1),
    "sec_per_val_item": round(t_eval / VAL_EVAL_SIZE, 2),
    "train_loss_curve": train_losses,
    "eval_loss_curve": eval_losses,
    "metrics_final": metrics_final,
    "metrics_per_epoch": metrics_per_epoch,
    "samples_20": samples_out,
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved: {RESULTS_FILE}")

Saved: results/v1_results.json


In [24]:
# Push adapter + tokenizer len HF (Q7, Q4=A)
print(f"Pushing adapter to {HF_REPO_ADAPTER} (private)...")
model.push_to_hub(HF_REPO_ADAPTER, private=True)
tokenizer.push_to_hub(HF_REPO_ADAPTER, private=True)
print(f"Pushed: https://huggingface.co/{HF_REPO_ADAPTER}")

Pushing adapter to SeanSunny/qwen3.5-4b-vn-pricer-v1 (private)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed: https://huggingface.co/SeanSunny/qwen3.5-4b-vn-pricer-v1


## Leaderboard Day 5

In [25]:
v0_rmsle = 4.4428
v1_rmsle = metrics_final["rmsle"]

print(f"{'Version':<30} {'RMSLE':>8} {'MAE':>14} {'MAPE':>8} {'R2':>8}")
print("-" * 72)
print(f"{'v0 zero-shot':<30} {v0_rmsle:>8.4f} {'296,807':>14} {'105.9%':>8} {'-2.0932':>8}")
for ep_key, m in metrics_per_epoch.items():
    label = f"v1 smoke ({ep_key})"
    print(f"{label:<30} {m['rmsle']:>8.4f} {m['mae']:>14,.0f} {m['mape']:>7.1f}% {m['r2']:>8.4f}")
print(f"{'v1 smoke (final epoch 2)':<30} {v1_rmsle:>8.4f} "
      f"{metrics_final['mae']:>14,.0f} {metrics_final['mape']:>7.1f}% {metrics_final['r2']:>8.4f}")
print(f"{'v8 Day4 (ref)':<30} {'0.4004':>8} {'79,853':>14} {'30.7%':>8} {'0.6920':>8}")
print()
print(f"Improvement v0 -> v1 final: {v0_rmsle - v1_rmsle:+.4f}")
print()
print("Buoc tiep: Phase 3 — Full train v2 (04_train_v2.ipynb)")

Version                           RMSLE            MAE     MAPE       R2
------------------------------------------------------------------------
v0 zero-shot                     4.4428        296,807   105.9%  -2.0932
v1 smoke (epoch_1)               0.6295        122,374    50.5%   0.3409
v1 smoke (epoch_2)               0.6084        116,769    50.4%   0.3980
v1 smoke (final epoch 2)         0.6084        116,769    50.4%   0.3980
v8 Day4 (ref)                    0.4004         79,853    30.7%   0.6920

Improvement v0 -> v1 final: +3.8344

Buoc tiep: Phase 3 — Full train v2 (04_train_v2.ipynb)
